# Практика: PDP

В уроке о PDP мы определили частичную зависимость как матожидание прогноза по распределению остальных признаков, посчитали ее руками на пяти строках и разобрали ее ловушки: обманчивую ось, плоскую кривую при разнонаправленных эффектах и подстановку значений, которых в данных не бывает. В этой тетради PDP строится руками на настоящих данных, сверяется с `scikit-learn`, и каждая ловушка воспроизводится числом.

В каждом задании есть ячейка с `...` — ее нужно дописать. Заданий в тренажере по этой тетради нет.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence
from sklearn.inspection import PartialDependenceDisplay
from sklearn.neighbors import NearestNeighbors
%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Данные и модель

Набор California Housing встроен в `sklearn`: 20 640 районов Калифорнии из переписи 1990 года, восемь признаков района, целевая переменная — медианная стоимость дома в районе в сотнях тысяч долларов. Признаки: `MedInc` — медианный доход (десятки тысяч долларов), `HouseAge` — медианный возраст дома, `AveRooms` и `AveBedrms` — среднее число комнат и спален на дом, `Population` — население района, `AveOccup` — среднее число жильцов на дом, `Latitude` и `Longitude` — широта и долгота.

Делим данные на обучающую и тестовую части (80% и 20%, `random_state=42`) и обучаем градиентный бустинг `HistGradientBoostingRegressor` с параметрами по умолчанию. Для методов этой тетради модель — черный ящик: от нее нужен только прогноз `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"R^2 модели на тестовых данных: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. PDP руками

$$\text{PD}(x_s) \approx \frac{1}{n}\sum_{i=1}^{n}\hat f\bigl(x_s, x_c^{(i)}\bigr).$$

Для каждого значения сетки подставляем его в столбец признака во всех $n$ строках, делаем прогноз и усредняем.

**Задание 1.** Допишите `pdp_by_hand` и сверьте результат на 500 тестовых строках с `partial_dependence(..., method="brute")`. Затем посчитайте ту же кривую методом по умолчанию для бустинга, `method="recursion"`, и посмотрите, чем она отличается.

In [ ]:
def pdp_by_hand(model, X, feature, grid):
    ...  # ваш код: len(grid) копий X, в копии t столбец feature равен grid[t]; прогноз и среднее по строкам каждой копии
    return ...


X_pdp = X_test.sample(500, random_state=0)
brute = partial_dependence(model, X_pdp, ["MedInc"], kind="average", method="brute", grid_resolution=20)
grid = brute["grid_values"][0]
pd_hand = pdp_by_hand(model, X_pdp, "MedInc", grid)
print("Наибольшее расхождение с brute:", np.abs(pd_hand - brute["average"][0]).max())

rec = partial_dependence(model, X_pdp, ["MedInc"], kind="average", method="recursion", grid_resolution=20)
shift = brute["average"][0] - rec["average"][0]
print("brute минус recursion: от", shift.min().round(3), "до", shift.max().round(3))
print("Среднее целевой переменной на обучающих данных:", y_train.mean().round(3))

**Место для ваших выводов:** совпала ли ваша кривая с `brute`? На сколько и почему `recursion` сдвинута относительно `brute`? Подсказка: `recursion` не проходит по строкам `X`, а спускается по деревьям и не учитывает начальный прогноз бустинга.

## 2. Стоимость PDP

Главная стоимость PDP — число прогнозов: каждое из $K$ значений сетки подставляется во все $n$ строк.

**Задание 2.** Оберните модель в счетчик строк, которые она прогнозирует, и проверьте, что PDP по сетке из $K = 20$ точек на $n = 500$ строках стоит $K \cdot n$ прогнозов.

In [ ]:
class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        ...  # ваш код: прибавить к счетчику число строк X
        return self.model.predict(X)


counted = CountingModel(model)
pdp_by_hand(counted, X_pdp, "MedInc", grid)
print("Прогнозов:", counted.rows, " K * n =", len(grid) * len(X_pdp))

## 3. PDP по нескольким признакам и по паре признаков

**Задание 3.** Постройте `PartialDependenceDisplay` для признаков `MedInc`, `AveOccup`, `HouseAge`, `AveRooms` и для пары (`AveOccup`, `HouseAge`) на тех же 500 строках, `method="brute"`. Какой признак сдвигает средний прогноз сильнее всего? Где на двумерной карте средний прогноз самый высокий?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
display = ...  # ваш код: PartialDependenceDisplay.from_estimator по четырем признакам и паре
for a in display.axes_.ravel():
    if a is not None and a.get_ylabel() == "Partial dependence":
        a.set_ylabel("частичная зависимость")
fig.suptitle("PDP стоимости дома: California Housing, градиентный бустинг, 500 тестовых строк")
fig.tight_layout()
plt.show()

## 4. Ловушка оси

PDP показывает средний прогноз, и при автомасштабе оси даже малое изменение выглядит крутым подъемом.

**Задание 4.** Нарисуйте PDP по `HouseAge` на двух осях: с автомасштабом и в пределах всего диапазона прогнозов модели на тестовых данных (от минимума до максимума). Посчитайте, какую долю этого диапазона занимает размах PDP.

In [ ]:
g_age = np.linspace(X_train["HouseAge"].quantile(0.05), X_train["HouseAge"].quantile(0.95), 20)
pd_age = pdp_by_hand(model, X_pdp, "HouseAge", g_age)
pred = model.predict(X_test)
share = ...  # ваш код: размах PDP, деленный на размах прогнозов
print(f"Размах PDP: {pd_age.max() - pd_age.min():.3f}, размах прогнозов: {pred.max() - pred.min():.3f}, доля: {share:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, lim in zip(axes, [None, (pred.min(), pred.max())]):
    ax.plot(g_age, pd_age, color=BLUE, lw=2)
    ax.set_xlabel("HouseAge")
    ax.set_ylabel("прогноз, сотни тысяч долларов")
    if lim:
        ax.set_ylim(*lim)
axes[0].set_title("автомасштаб")
axes[1].set_title("весь диапазон прогнозов")
plt.show()

## 5. PDP-based Feature Importance

Для непрерывного признака метрика — стандартное отклонение кривой PDP по $k$ точкам сетки:

$$I_{x_1} = \sqrt{\frac{1}{k-1}\sum_{i=1}^{k}\left(f(x_{1i}) - \frac{1}{k}\sum_{j=1}^{k}f(x_{1j})\right)^{2}}.$$

**Задание 5.** Посчитайте $I$ для всех восьми признаков по сетке из 20 точек между 5-м и 95-м процентилями признака и упорядочьте признаки по убыванию.

In [ ]:
def pdp_fi(pd_curve):
    return ...  # ваш код: стандартное отклонение кривой с делителем k - 1


fi = {}
for feature in X.columns:
    g = np.linspace(X_train[feature].quantile(0.05), X_train[feature].quantile(0.95), 20)
    fi[feature] = pdp_fi(pdp_by_hand(model, X_pdp, feature, g))
pd.Series(fi, name="PDP-based FI").sort_values(ascending=False).round(3)

## 6. Плоский PDP при сильном эффекте

Соберем синтетические данные, где у половины объектов прогноз растет с признаком, а у другой половины падает: $y = 3\,s\,x_1 + \varepsilon$, где $x_1 \sim U(-1, 1)$, $s \in \{-1, +1\}$ — признак группы, $\varepsilon \sim N(0, 0.1^2)$. Обучим на $(x_1, s)$ тот же бустинг.

**Задание 6.** Постройте PDP по $x_1$ и посчитайте для него PDP-based FI. Затем нарисуйте ICE-кривые 60 объектов, раскрасив их по группе $s$, и посчитайте средний по объектам разброс отдельной ICE-кривой (стандартное отклонение кривой по сетке).

In [ ]:
rng = np.random.RandomState(0)
n = 2000
toy = pd.DataFrame({"x1": rng.uniform(-1, 1, n), "s": rng.choice([-1, 1], n)})
y_toy = 3 * toy["s"] * toy["x1"] + rng.normal(0, 0.1, n)
toy_model = HistGradientBoostingRegressor(random_state=0).fit(toy, y_toy)

g_toy = np.linspace(-0.95, 0.95, 20)
toy_ice = toy.iloc[:60]
X_rep = pd.concat([toy_ice] * len(g_toy), ignore_index=True)
X_rep["x1"] = np.repeat(g_toy, len(toy_ice))
ice_toy = toy_model.predict(X_rep).reshape(len(g_toy), len(toy_ice)).T
pd_toy = ...      # ваш код: PDP по x1 на всех строках toy
ice_spread = ...  # ваш код: средний разброс отдельной ICE-кривой
print(f"PDP-based FI(x1) = {pdp_fi(pd_toy):.3f}, средний разброс ICE-кривой = {ice_spread:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for s, color in [(-1, ORANGE), (1, BLUE)]:
    ax.plot(g_toy, ice_toy[toy_ice["s"].values == s].T, color=color, alpha=0.4, lw=1)
ax.plot(g_toy, pd_toy, color=INK, lw=3, label="PDP")
ax.set_xlabel("x1")
ax.set_ylabel("прогноз")
ax.set_title("ICE 60 объектов (синие: s = +1, оранжевые: s = -1) и PDP")
ax.legend()
plt.show()

**Место для ваших выводов:** что говорит PDP-based FI о признаке $x_1$ и что — ICE?

## 7. Двумерный PDP и подстановка нереалистичных значений

Широта и долгота в California Housing связаны: районы вытянуты вдоль побережья, и большая часть прямоугольника (широта $\times$ долгота) приходится на океан и пустыни. PDP подставляет каждую пару значений сетки во все строки, не глядя, бывает ли такой район.

**Задание 7.** Постройте двумерный PDP по паре (`Longitude`, `Latitude`) на 300 тестовых строках, сетка $20 \times 20$, и отметьте на карте настоящие районы. Затем посчитайте долю узлов сетки, рядом с которыми нет ни одного района: ближайший район из всего набора дальше 0.25 градуса.

In [ ]:
X_geo = X_test.sample(300, random_state=0)
res = partial_dependence(model, X_geo, ["Longitude", "Latitude"], kind="average", method="brute", grid_resolution=20)
lon, lat = res["grid_values"]
surface = res["average"][0]

nodes = np.array([(a, b) for a in lon for b in lat])
nn = NearestNeighbors(n_neighbors=1).fit(X[["Longitude", "Latitude"]].values)
dist = nn.kneighbors(nodes)[0][:, 0]
empty_share = ...  # ваш код: доля узлов, у которых ближайший район дальше 0.25
print(f"Узлов сетки без района ближе 0.25 градуса: {(dist > 0.25).sum()} из {len(nodes)} ({empty_share:.1%})")

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(lon, lat, surface.T, levels=12, cmap="Blues")
fig.colorbar(cs, ax=ax, label="частичная зависимость, сотни тысяч долларов")
ax.scatter(X["Longitude"], X["Latitude"], s=1, color=ORANGE, alpha=0.3, label="настоящие районы")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Двумерный PDP по долготе и широте: California Housing, градиентный бустинг", fontsize=10)
ax.legend(loc="upper right", markerscale=8)
plt.show()

**Место для ваших выводов:** какая доля карты PDP посчитана по районам, которых не существует? Что это значит для чтения двумерного PDP и почему следующий урок модуля переходит к ALE?